## Import

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape

## Path setup

In [ ]:
from pathlib import Path
data_dir = Path("data/unzipped/")
geojson_out = data_dir / "geojson"
parquet_out = data_dir / "geoparquet"
geojson_out.mkdir(exist_ok=True)
parquet_out.mkdir(exist_ok=True)

## Convert to GeoJSON (test)

In [13]:
def load_jsonl_to_gdf(path, nrows=None):
    """
    Read a Microsoft building tile (JSON Lines) into a GeoDataFrame.
    Set nrows to read only a preview subset.
    """
    df = pd.read_json(path, lines=True, nrows=nrows)
    # Convert dict -> shapely geometry
    df["geometry"] = df["geometry"].apply(shape)
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs=4326)  # WGS84
    return gdf

In [14]:
# Convert 1 file to GeoJSON to open it in QGIS
sample_files = sorted(data_dir.glob("part-*.csv"))[:1]  # first one tile
for p in sample_files:
    gdf = load_jsonl_to_gdf(p, nrows=10000)  # cap rows for a lighter file
    out_path = geojson_out / f"{p.stem}.geojson"
    gdf.to_file(out_path, driver="GeoJSON")
    print("Wrote:", out_path)

Wrote: data\unzipped\geojson\part-00012-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.geojson


In [17]:
g = gpd.read_file(geojson_out / f"{sample_files[0].stem}.geojson")
g.head()

,type,properties,geometry
0,Feature,"{ ""height"": -1.0, ""confidence"": 0.923900008201...","POLYGON ((88.03805 24.68878, 88.03805 24.68886..."
1,Feature,"{ ""height"": -1.0, ""confidence"": 0.952000021934...","POLYGON ((88.03442 24.66351, 88.03452 24.6635,..."
2,Feature,"{ ""height"": -1.0, ""confidence"": 0.971400022506...","POLYGON ((88.03813 24.6892, 88.03799 24.68919,..."
3,Feature,"{ ""height"": -1.0, ""confidence"": 0.952199995517...","POLYGON ((88.03616 24.66548, 88.03624 24.66547..."
4,Feature,"{ ""height"": -1.0, ""confidence"": 0.958100020885...","POLYGON ((88.03847 24.68029, 88.03845 24.68032..."


## Convert all to GeoParquet

In [16]:
# Full conversion to GeoParquet (one parquet per tile)
for p in sorted(data_dir.glob("part-*.csv")):
    gdf = load_jsonl_to_gdf(p)
    out_path = parquet_out / f"{p.stem}.parquet"
    gdf.to_parquet(out_path, index=False)  # GeoParquet (CRS + geometry preserved)
    print("Wrote:", out_path)

Wrote: data\unzipped\geoparquet\part-00012-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
Wrote: data\unzipped\geoparquet\part-00013-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
Wrote: data\unzipped\geoparquet\part-00015-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000 (1).parquet
Wrote: data\unzipped\geoparquet\part-00015-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
Wrote: data\unzipped\geoparquet\part-00019-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
Wrote: data\unzipped\geoparquet\part-00023-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
Wrote: data\unzipped\geoparquet\part-00026-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000 (1).parquet
Wrote: data\unzipped\geoparquet\part-00026-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
Wrote: data\unzipped\geoparquet\part-00027-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
Wrote: data\unzipped\geoparquet\part-00028-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
Wrote: data\unzipped\geoparquet\part-00033-5cf70943-9c5f-4fc6-

# querying

In [2]:
from pathlib import Path
import geopandas as gpd
# import pandas as pd
from shapely.geometry import Point, Polygon
from shapely.ops import transform as shp_transform
from pyproj import Transformer

## Load GeoParquet files

In [3]:
PARQUET_DIR = Path("data/unzipped/geoparquet")

PARQUET_FILES = sorted(PARQUET_DIR.glob("*.parquet"))
len(PARQUET_FILES)

53

### Helpers: order points and pick a good CRS (auto‑UTM)

In [4]:
import math

def order_points_ccw_latlon(points_latlon):
    """
    Require input as (lat, lon) pairs. Returns polygon vertices as (lon, lat) ordered CCW.
    """
    pts = [(float(lon), float(lat)) for (lat, lon) in points_latlon]
    cx = sum(p[0] for p in pts) / len(pts)
    cy = sum(p[1] for p in pts) / len(pts)
    pts = sorted(pts, key=lambda p: math.atan2(p[1] - cy, p[0] - cx))
    poly = Polygon(pts)
    if not poly.exterior.is_ccw:
        pts.reverse()  # ensure CCW
    return pts

def utm_epsg_for_lonlat(lon, lat):
    """
    Returns an EPSG code for the UTM zone matching (lon, lat).
    N hemisphere adds 32600, S hemisphere adds 32700.
    Bangladesh spans UTM 45N–46N–47N; this will pick the right one by centroid.
    """
    zone = int(math.floor((lon + 180) / 6) + 1)
    base = 32600 if lat >= 0 else 32700
    return base + zone


### Core function: total area inside a polygon (any 4 points)

In [5]:
def total_building_area_polygon(points_latlon, parquet_files=PARQUET_FILES, verbose=True):
    """
    points_latlon: list of (lat, lon) pairs clicked on the map (any order).
    Computes total building footprint area (m²) within the polygon.
    """
    # 1) Order and build polygon in WGS84
    pts_lonlat = order_points_ccw_latlon(points_latlon)   # requires (lat, lon)
    aoi_4326 = Polygon(pts_lonlat).buffer(0)
    if aoi_4326.is_empty:
        raise ValueError("AOI polygon invalid/empty after repair.")

    # 2) Choose UTM CRS for accurate areas
    c_lon, c_lat = aoi_4326.centroid.x, aoi_4326.centroid.y
    epsg = utm_epsg_for_lonlat(c_lon, c_lat)
    to_utm = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True).transform
    aoi_utm = shp_transform(to_utm, aoi_4326)

    # 3) Coarse prefilter using AOI bbox in 4326
    minx, miny, maxx, maxy = aoi_4326.bounds

    total_area_m2 = 0.0
    total_count = 0
    touched = 0

    for i, fp in enumerate(parquet_files, 1):
        if verbose and i % 5 == 0:
            print(f"[{i}/{len(parquet_files)}] {fp.name}")

        gdf = gpd.read_parquet(fp)
        if gdf.crs is None:
            gdf = gdf.set_crs(4326)

        sub_bbox = gdf.cx[minx:maxx, miny:maxy]
        if sub_bbox.empty:
            continue

        sub_utm = sub_bbox.to_crs(epsg=epsg)
        mask = sub_utm.intersects(aoi_utm)
        if not mask.any():
            continue

        inside = sub_utm.loc[mask]
        total_area_m2 += float(inside.area.sum())
        total_count += int(len(inside))
        touched += 1

    return {
        "epsg_used": epsg,
        "tiles_used": touched,
        "building_count": total_count,
        "total_area_m2": total_area_m2,
    }

### Run 

In [6]:
# polygon coordinates (replace with your 4 clicks):
coords = [
    (23.896029855616522, 90.35975405908485),  # (lat, lon)
    (23.900621847847283, 90.45384411368158),
    (23.711296988178443, 90.36109341936216),
    (23.671522599900555, 90.51081016270012),
]

res = total_building_area_polygon(coords, verbose=True)
res

[5/53] part-00019-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
[10/53] part-00028-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
[15/53] part-00056-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
[20/53] part-00077-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
[25/53] part-00093-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
[30/53] part-00112-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
[35/53] part-00126-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
[40/53] part-00160-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
[45/53] part-00172-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet
[50/53] part-00196-5cf70943-9c5f-4fc6-94fb-43ce5feefa56.c000.parquet


{'epsg_used': 32646,
 'tiles_used': 2,
 'building_count': 178258,
 'total_area_m2': 68569018.51751679}

In [9]:
area = res['total_area_m2']
print("Total Rooftop Area in meter square:", area)

Total Rooftop Area in meter square: 68569018.51751679


In [10]:
solar_area = 1.03
solar_panel_count = area/solar_area
print("Total Count of Solar Panel:", solar_panel_count)

Total Count of Solar Panel: 66571862.63836581


In [11]:
price_per_m2 = 12300
total_cost = price_per_m2*area
print("Total Cost: ", total_cost)

Total Cost:  843398927765.4565


In [12]:
generated_energy_per_m2 = 0.2      #kwh/m^2/h
total_generated_energy = generated_energy_per_m2*area
print(f"Estimated solar energy (1 hour): {total_generated_energy:.2f} kWh") # total energy generated in 1hr
# assuming 5hr/day runtime
monthly= total_generated_energy*5*30
yearly = monthly*12
print(f"Estimated solar energy (1 month): {monthly:.2f} kWh")
print(f"Estimated solar energy (1 year): {yearly:.2f} kWh") 

Estimated solar energy (1 hour): 13713803.70 kWh
Estimated solar energy (1 month): 2057070555.53 kWh
Estimated solar energy (1 year): 24684846666.31 kWh
